# WavLM Real Audio Training — Colab GPU

Trains on REAL audio from GDrive on GPU.
Target: F1 >= 0.55

**Runtime → Change runtime type → GPU → Run All**

**NOTE:** This notebook uses batch1/batch2 segments (128K+ samples) that have audio on GDrive.

In [ ]:
# Setup
!pip install -q transformers librosa soundfile accelerate

import torch
import numpy as np
import json
import time
import os
import subprocess
import librosa
from collections import Counter
from transformers import WavLMModel
from sklearn.metrics import f1_score, precision_score, recall_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)
print("GDrive mounted!")

In [ ]:
# Build video_id → audio path mapping for GDrive audio
def list_gdrive_videos(folder):
    """List video IDs in a GDrive audio subfolder."""
    result = subprocess.run(
        ["rclone", "lsjson", f"gdrive:laughter_prediction/audio/{folder}/"],
        capture_output=True, text=True, timeout=30
    )
    try:
        files = json.loads(result.stdout)
        return {f['Path'].replace(f'{folder}/', '').replace('.mp3', ''): f'{folder}/' + f['Path'].split('/')[-1] for f in files if f['Path'].endswith('.mp3')}
    except:
        return {}

print("Scanning GDrive audio folders...")
en_map = list_gdrive_videos("en")
print(f"  en: {len(en_map)} videos")
zh_map = list_gdrive_videos("zh")
print(f"  zh: {len(zh_map)} videos")
hi_map = list_gdrive_videos("hi-latn")
print(f"  hi-latn: {len(hi_map)} videos")

# Combine into one mapping: video_id → GDrive path
GDrive_AUDIO_MAP = {}
GDrive_AUDIO_MAP.update({vid: f"en/{fname}" for vid, fname in en_map.items()})
GDrive_AUDIO_MAP.update({vid: f"zh/{fname}" for vid, fname in zh_map.items()})
GDrive_AUDIO_MAP.update({vid: f"hi-latn/{fname}" for vid, fname in hi_map.items()})
print(f"\nTotal GDrive audio videos: {len(GDrive_AUDIO_MAP)}")

In [ ]:
# Load segments from batch1 and batch2, filtering to only those with GDrive audio
BASE = "/content/gdrive/MyDrive/laughter_prediction"
MAX_SAMPLES = 80000
SR = 16000
samples = []

def load_segments_from_file(filepath, max_lines):
    """Load segments from a jsonl file, only returning those with GDrive audio."""
    loaded = 0
    found = 0
    local_samples = []
    
    try:
        with open(filepath) as f:
            for line in f:
                if loaded >= max_lines:
                    break
                loaded += 1
                try:
                    d = json.loads(line)
                    vid = d.get('video_id', '')
                    start = d.get('start')
                    end = d.get('end')
                    label = int(d.get('label', 0)) if str(d.get('label', '0')).isdigit() else 0
                    
                    # Only include if we have this video on GDrive
                    if vid in GDrive_AUDIO_MAP:
                        gdrive_rel = GDrive_AUDIO_MAP[vid]
                        full_path = f"{BASE}/audio/{gdrive_rel}"
                        if os.path.exists(full_path) and start is not None and end is not None and end > start:
                            local_samples.append({
                                "audio_file": full_path,
                                "start": start,
                                "end": end,
                                "label": label
                            })
                            found += 1
                except:
                    pass
    except FileNotFoundError:
        print(f"  File not found: {filepath}")
    
    return local_samples

print("Loading batch1 segments...")
batch1 = load_segments_from_file(f"{BASE}/aligned/batch1/aligned_segments_batch1.jsonl", MAX_SAMPLES)
print(f"  Loaded {len(batch1)} segments with GDrive audio")
print("Loading batch2 segments...")
batch2 = load_segments_from_file(f"{BASE}/aligned/batch2/aligned_segments_batch2.jsonl", MAX_SAMPLES)
print(f"  Loaded {len(batch2)} segments with GDrive audio")

samples = batch1 + batch2
print(f"\nTotal: {len(samples)} segments with audio")

labels = Counter(s["label"] for s in samples)
print(f"Laugh: {labels[1]} ({100*labels[1]/len(samples):.1f}%)")
print(f"No laugh: {labels[0]} ({100*labels[0]/len(samples):.1f}%)")

In [ ]:
# Audio loading with LRU cache
from functools import lru_cache

@lru_cache(maxsize=32)
def load_audio_cached(path):
    try:
        y, _ = librosa.load(path, sr=SR, mono=True)
        return y
    except Exception as e:
        return None

def extract_segment(path, start, end, pad_ms=50):
    y = load_audio_cached(path)
    if y is None:
        return np.zeros(int(0.3 * SR), dtype=np.float32)
    s = int(max(0, start - pad_ms/1000) * SR)
    e = int(min(len(y)/SR, end + pad_ms/1000) * SR)
    if s >= len(y) or e <= s:
        return np.zeros(int(0.3 * SR), dtype=np.float32)
    seg = y[s:e]
    if len(seg) < int(0.1 * SR):
        seg = np.pad(seg, (0, int(0.1 * SR) - len(seg)))
    return seg.astype(np.float32)

# Test
test = extract_segment(samples[0]["audio_file"], samples[0]["start"], samples[0]["end"])
print(f"Audio test: shape={test.shape}, duration={len(test)/SR:.3f}s")
print(f"Path: {samples[0]['audio_file']}")

In [ ]:
# Load WavLM-Base+
print("Loading WavLM-Base+ (94M params)...")
t0 = time.time()
encoder = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
encoder.to(device)
encoder.eval()
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Model: frozen WavLM + MLP head
class WavLMLaughterClassifier(torch.nn.Module):
    def __init__(self, enc):
        super().__init__()
        self.encoder = enc
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 256), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(256, 64), torch.nn.ReLU(), torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 2)
        )
    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x).last_hidden_state.mean(dim=1)
        return self.classifier(h)

model = WavLMLaughterClassifier(encoder).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

In [ ]:
# Training
BATCH, EPOCHS, LR = 64, 5, 1e-3
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([1., 5.]).to(device))

print(f"Training: {EPOCHS} epochs, batch={BATCH}, samples={len(samples)}, device={device}")

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    preds, ys = [], []
    nb = (len(samples) + BATCH - 1) // BATCH
    model.train()
    
    for bi in range(nb):
        batch = samples[bi*BATCH : (bi+1)*BATCH]
        
        # Load real audio
        audio_batch = [extract_segment(s["audio_file"], s["start"], s["end"]) for s in batch]
        max_len = max(len(a) for a in audio_batch)
        padded = np.zeros((len(audio_batch), max_len), dtype=np.float32)
        for i, a in enumerate(audio_batch):
            padded[i, :len(a)] = a
        
        x = torch.from_numpy(padded).to(device)
        y = torch.tensor([s["label"] for s in batch]).to(device)
        
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        
        preds.extend(model(x).argmax(1).cpu().tolist())
        ys.extend(y.cpu().tolist())
        
        if (bi+1) % 100 == 0:
            print(f"  Batch {bi+1}/{nb}, loss={loss.item():.4f}")
    
    f1 = f1_score(ys, preds, zero_division=0)
    p = precision_score(ys, preds, zero_division=0)
    r = recall_score(ys, preds, zero_division=0)
    print(f"Epoch {ep}/{EPOCHS} ({time.time()-t0:.0f}s) F1={f1:.4f} P={p:.4f} R={r:.4f}")

In [ ]:
# Save
save_path = "/content/gdrive/MyDrive/wavlm_phaseA_colab.pt"
torch.save(model.state_dict(), save_path)
print(f"Saved: {save_path}")